# Analysis of the Allegheny County Jail Oversight Board Meeting Minutes

- Contributor: Danielle Aira Savellano
- AI Acknowledgements: The following code was written with assistance from GitHub Copilot.

## Text Preprocessing — PDF/DOC/DOCX to TXT

This notebook contains code for [downloading files](#i-file-downloading) and [extracting text](#ii-text-extraction).

### [File Downloading](#i-file-downloading)

Meeting minutes were downloaded from the [JOB website](https://alleghenycontroller.com/the-controller/jail-oversight-board/jail-oversight-board-internal/) as `PDF` or `DOCX` files for consistency. A CSV was created using Excel for desired filenames and links for download. This section contains the code used to:

- Download from `Data/Scraped/download_links.csv` (`Desired Title`, `Link`, `Year`, `Mo`) into `Data/Scraped` with the desired filename and extension
- Write `Data/Scraped/download_tracker.csv` with the same columns plus `Downloaded` status and `File Type`

A total of **156 files were downloaded**. There were 116 PDFs and 40 DOCX files. Meetings ranged from March 2012 to February 2026. Furthermore, there were 4 special meetings (additional to monthly meetings).

> Files were named `YYYY_M_X_JOB_Minutes` where `YYYY` = year, `M` = Mo, and `X` = special meeting or not. For example: `2024_2_1_JOB_Minutes` → February 2024 Special Meeting

A summary by file-type, year, and special meeting status can be found at the end of the downloading section.

### [Text Extraction](#ii-text-extraction)

After the source files were downloaded, they were converted into .TXT files under `Data/Text/`. Code was also added to load .TXT files for Python use.

**153 out of 156 files were successfully extracted into text.** However, 3 files resulted in errors wherein blank text files were generated. Error Files were uploaded to a folder `Data/Error_Files/`. Upon inspection, errors were likely due to scanning quality.

Furthermore, some inconsistencies were identified, particularly in the more recent minutes. These minutes included glossaries, tables, and line numbering, which did not convert properly. With the focus on discussions, it was determined that the transcript text will be sufficient and so tables should not be a focus.

In [1]:
!pip install pypdf python-docx --quiet


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip install --upgrade pip


### I. File Downloading

In [2]:
import csv
import time
from pathlib import Path
from urllib.parse import urlparse

import requests

# Paths and Session Setup
BASE = Path("..").resolve()
CSV_IN = BASE / "Data" / "Scraped" / "download_links.csv"
OUT_DIR = BASE / "Data" / "Scraped"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_OUT = OUT_DIR / "download_tracker.csv"
sess = requests.Session()

# Helper Functions
def suffix(url: str) -> str:
    '''Extract file suffix from URL, default to .pdf if not found.'''
    s = Path(urlparse(url).path).suffix.lower()
    return s if s else ".pdf"

def desired_path(title: str, ext: str) -> Path:
    '''Generate desired path given title and extension.'''
    ext = ext if ext.startswith(".") else f".{ext}"
    if ext == ".doc":
        ext = ".docx"  # Convert .doc to .docx
    return OUT_DIR / f"{title}{ext.lower()}"

# Prepare output fields and results list
with CSV_IN.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    cols = list(reader.fieldnames or [])
    rows_in = list(reader)

extra = ["Downloaded", "File Type"]
out_fields = cols + [c for c in extra if c not in cols]
results = []
n_skipped = 0

# Processing input CSV
for row in rows_in:
    title = (row.get("Desired Title")).strip()
    url = (row.get("Link")).strip()
    out = {**row}                                           # Keep original data
    ext = suffix(url)                                       # Get file extension

    target = desired_path(title, ext)                       # Desired path (combines title and extension)

    if target.is_file() and target.stat().st_size > 0:      # If file exists and is not empty
        n_skipped += 1
        out["Downloaded"], out["File Type"] = "True", target.suffix.lower()
        results.append(out)
        continue                                            # Skip download

    try:
        r = sess.get(url, timeout=90)
        r.raise_for_status()
        target.write_bytes(r.content)
        out["Downloaded"], out["File Type"] = "True", ext   # Add download status and file type
    except Exception as e:
        out["Downloaded"], out["File Type"] = "False", ext
        print(f"{title}: {e}")

    results.append(out)                                     # Append result for document
    time.sleep(0.2)

# Write results to output CSV
with CSV_OUT.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=out_fields, extrasaction="ignore")
    w.writeheader()
    w.writerows(results)

# Summary
downloaded = sum(1 for r in results if r.get("Downloaded") == "True")
print(f"{downloaded}/{len(results)} Downloaded ({n_skipped} Already Existing)")
print(f"Saved to: {OUT_DIR}")
print(f"Tracker: {CSV_OUT}")

156/156 Downloaded (6 Already Existing)
Saved to: /Users/daesavellano/0_CMU/Unstructured Data Analytics/Project/Updated/board_meeting_analysis/Data/Scraped
Tracker: /Users/daesavellano/0_CMU/Unstructured Data Analytics/Project/Updated/board_meeting_analysis/Data/Scraped/download_tracker.csv


In [3]:
# Download Summary

print(f"\n{len(results)} Total Files")

print("\nFile Types")
for file_type in set(r.get("File Type") for r in results):
    count = sum(1 for r in results if r.get("File Type") == file_type)
    print(f"{file_type}: {count}")

print("\nYears")

for year in sorted(set(r.get("Year") for r in results)):
    count = sum(1 for r in results if r.get("Desired Title").split("_")[0] == year)
    print(f"{year}: {count}")


156 Total Files

File Types
.doc: 37
.docx: 3
.pdf: 116

Years
2012: 7
2013: 8
2014: 10
2015: 9
2016: 11
2017: 12
2018: 12
2019: 11
2020: 10
2021: 13
2022: 12
2023: 12
2024: 14
2025: 13
2026: 2


In [4]:
special_count = 0

print("\nSpecial Meetings:")
for r in results:
    if r.get("Desired Title").split("_")[2] == "1":
        special_count += 1
        print(r.get("Desired Title"))

print(f"\nTotal Special Meetings: {special_count}")


Special Meetings:
2021_9_1_JOB_Minutes
2024_2_1_JOB_Minutes
2024_3_1_JOB_Minutes
2025_8_1_JOB_Minutes

Total Special Meetings: 4


### II. Text Extraction

In [5]:
from pathlib import Path

# Paths
BASE = Path("..").resolve()
IN_DIR = BASE / "Data" / "Scraped"
OUT_DIR = BASE / "Data" / "Text"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Functions to read, extract, and convert files to plain text.

def read_text_file(path: Path) -> str:
    for encoding in ("utf-8", "utf-8-sig"):
        try:
            return path.read_text(encoding=encoding)
        except UnicodeDecodeError:
            continue
    return path.read_text(encoding="utf-8", errors="replace")

def extract_pdf_to_text(path: Path) -> str:
    '''Extract text from a PDF file using pypdf. Written by GitHub Copilot.'''
    from pypdf import PdfReader
    # strict=False tolerates slightly malformed PDFs from some scanners/tools.
    reader = PdfReader(str(path), strict=False)
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n\n".join(pages).strip()

def extract_docx_to_text(path: Path) -> str:
    '''Extract text from a docx file using python-docx. Written by GitHub Copilot.'''
    from docx import Document
    doc = Document(str(path))
    paragraphs = [p.text for p in doc.paragraphs]
    return "\n\n".join(paragraphs).strip()

def file_to_plain_text(path: Path) -> str:
    if path.suffix.lower() == ".pdf":
        return extract_pdf_to_text(path)
    if path.suffix.lower() == ".docx":
        return extract_docx_to_text(path)
    return read_text_file(path)

def list_input_files(directory: Path) -> list[Path]:
    # Sorted for reproducible order; skip dotfiles (.DS_Store, etc.).
    return sorted(
        p for p in directory.iterdir()
        if p.is_file() and not p.name.startswith(".")
    )

# Convert each input file to .txt for analysis and skip if output already exists
written: list[Path] = []
n_skipped = 0

for path in list_input_files(IN_DIR):
    # Skip non-document files (e.g., .csv)
    if path.suffix.lower() == ".csv":
        continue
    
    # Skip if the output .txt already exists (avoid re-processing)

    out_path = OUT_DIR / f"{path.stem}.txt"
    
    if out_path.exists() and out_path.stat().st_size == 0:  # Empty file exists
        n_skipped += 1
        written.append(out_path)

        print(f"Warning: {path.stem} already exists but is empty.")
        
        # Make copy of input file in Data/Error_Files/ for manual review if needed.
        error_dir = BASE / "Data" / "Error_Files"
        error_dir.mkdir(parents=True, exist_ok=True)
        error_path = error_dir / path.name
        if not error_path.exists():
            error_path.write_bytes(path.read_bytes())
        continue

    if out_path.exists() and out_path.stat().st_size > 0:
       n_skipped += 1
       written.append(out_path)
       continue                                            # Skip download

    text = file_to_plain_text(path)
    out_path.write_text(text, encoding="utf-8", newline="\n")
    written.append(out_path)

# Summary
print(f"\n{len(written)}/{len(list_input_files(IN_DIR))-2} Files Processed ({n_skipped} Already Existing)")
        
print(f"\nSaved to: {OUT_DIR}")


156/156 Files Processed (156 Already Existing)

Saved to: /Users/daesavellano/0_CMU/Unstructured Data Analytics/Project/Updated/board_meeting_analysis/Data/Text


In [6]:
# Load all converted .txt files for analysis

documents: list[str] = []                               # List of document texts
doc_names: list[str] = []                               # Corresponding list of filenames
for path in sorted(OUT_DIR.glob("*.txt")):
    documents.append(path.read_text(encoding="utf-8"))
    doc_names.append(path.name)

print(f"{len(documents)} document(s) → lists `documents` (text) and `doc_names` (filenames)")

156 document(s) → lists `documents` (text) and `doc_names` (filenames)


In [7]:
# Save all documents in a pickle file for later use in BERT topic modeling and analysis. Include month and year as separate columns for easier analysis.

import pickle
import re

# Creating file path to new "Combined" folder
path = BASE / "Data" / "Combined" / "raw_documents.pkl"
path.parent.mkdir(parents=True, exist_ok=True)

# Extract year and month from filenames (format: YYYY_M_X_JOB_Minutes.txt)
years = []
months = []
for name in doc_names:
    match = re.match(r'(\d{4})_(\d+)_', name)
    if match:
        years.append(int(match.group(1)))
        months.append(int(match.group(2)))
    else:
        years.append(None)
        months.append(None)

# Save documents along with metadata
data = {
    'documents': documents,
    'doc_names': doc_names,
    'years': years,
    'months': months
}

with open(path, 'wb') as f:
    pickle.dump(data, f)

print(f"Saved {len(documents)} documents with metadata to: {path}")

Saved 156 documents with metadata to: /Users/daesavellano/0_CMU/Unstructured Data Analytics/Project/Updated/board_meeting_analysis/Data/Combined/raw_documents.pkl
